## Using nn.LSTM() 

We will use nn.LSTM() to simplify the __init__() and forward()

In [1]:
import torch
import torch.nn as nn
from torch.optim import Adam
import lightning as L
from torch.utils.data import TensorDataset, DataLoader


In [12]:
class LightningLSTM(L.LightningModule):
    def __init__(self):
        super().__init__()
        L.seed_everything(seed=42)
        self.lstm = nn.LSTM(input_size=1, hidden_size=1)

    def forward(self, input):
        input_trans = input.view(len(input), 1)
        lstm_out, _ = self.lstm(input_trans)

        # lstm_out has the short-term memory for all inputs. We make our prediction with the last one
        prediction = lstm_out[-1]
        return prediction
    
    def configure_optimizers(self):
        return Adam(self.parameters(), lr=0.1)
    
    def training_step(self, batch):
        input_i, label_i = batch
        output_i = self.forward(input_i[0])
        loss = (output_i - label_i)**2

        self.log("train_loss", loss, on_step=True, on_epoch=True)
        if (label_i == 0):
            self.log("out_0", output_i, on_step=True)
        else:
            self.log("out_1", output_i, on_step=True)
        return loss

In [13]:
model = LightningLSTM()
print("Before optimization parameters are...")
for name, param in model.named_parameters():
    print(name, param)

print("Comparisson between the observed and predicted values...")
print("Company A: Observed = 0, Predicted = ", model(torch.tensor([0., 0.5, 0.25, 1.])).detach())
print("Company B: Observed = 0, Predicted = ", model(torch.tensor([1., 0.5, 0.25, 1.])).detach())

Seed set to 42


Before optimization parameters are...
lstm.weight_ih_l0 Parameter containing:
tensor([[ 0.7645],
        [ 0.8300],
        [-0.2343],
        [ 0.9186]], requires_grad=True)
lstm.weight_hh_l0 Parameter containing:
tensor([[-0.2191],
        [ 0.2018],
        [-0.4869],
        [ 0.5873]], requires_grad=True)
lstm.bias_ih_l0 Parameter containing:
tensor([ 0.8815, -0.7336,  0.8692,  0.1872], requires_grad=True)
lstm.bias_hh_l0 Parameter containing:
tensor([ 0.7388,  0.1354,  0.4822, -0.1412], requires_grad=True)
Comparisson between the observed and predicted values...
Company A: Observed = 0, Predicted =  tensor([0.6675])
Company B: Observed = 0, Predicted =  tensor([0.6665])


In [14]:
inputs = torch.tensor([[0., 0.5, 0.25, 1.], [1., 0.5, 0.25, 1.]])
lables = torch.tensor([0., 1.])

dataset = TensorDataset(inputs, lables)
dataloader = DataLoader(dataset)

In [15]:
# We will use learning rate = 0.1 and 300 epoches

trainer = L.Trainer(max_epochs=300, log_every_n_steps=2)
trainer.fit(model, train_dataloaders=dataloader)
print("After optimization the parameters are...")
for name, param in model.named_parameters():
    print(name, param.data)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.



  | Name | Type | Params | Mode  | FLOPs
----------------------------------------------
0 | lstm | LSTM | 16     | train | 0    
----------------------------------------------
16        Trainable params
0         Non-trainable params
16        Total params
0.000     Total estimated model params size (MB)
1         Modules in train mode
0         Modules in eval mode
0         Total Flops
/Users/odysseasgeorgiades/Desktop/Neural Networks/.venv/lib/python3.14/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/odysseasgeorgiades/Desktop/Neural Networks/.venv/lib/python3.14/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 299: 100%|██████████| 2/2 [00:00<00:00, 260.22it/s, v_num=0]

`Trainer.fit` stopped: `max_epochs=300` reached.


Epoch 299: 100%|██████████| 2/2 [00:00<00:00, 167.43it/s, v_num=0]
After optimization the parameters are...
lstm.weight_ih_l0 tensor([[3.5364],
        [1.3869],
        [1.5390],
        [1.2488]])
lstm.weight_hh_l0 tensor([[5.2070],
        [2.9577],
        [3.2652],
        [2.0678]])
lstm.bias_ih_l0 tensor([-0.9143,  0.3724, -0.1815,  0.6376])
lstm.bias_hh_l0 tensor([-1.0570,  1.2414, -0.5685,  0.3092])


In [16]:
print("Comparisson between the observed and predicted values...")
print("Company A: Observed = 0, Predicted = ", model(torch.tensor([0., 0.5, 0.25, 1.])).detach())
print("Company B: Observed = 0, Predicted = ", model(torch.tensor([1., 0.5, 0.25, 1.])).detach())

Comparisson between the observed and predicted values...
Company A: Observed = 0, Predicted =  tensor([6.7842e-05])
Company B: Observed = 0, Predicted =  tensor([0.9809])


In [17]:
# And to visualize the graphs on terminal we run 'tensorboard --logdir=lightning_logs/' and then go to http://localhost:6006/ 